In [19]:
# importacmos librerías
import pandas as pd
import numpy as np
import glob
import geopandas as gpd
from shapely import wkt
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


In [20]:
# funciones auxiliares
def estandarizar_codi_barri(df, columna_barri):
    """Filtramos nulos y forzamos cod_barrio a entero (1-73) para merges."""
    df = df.dropna(subset=[columna_barri]).copy()
    df[columna_barri] = pd.to_numeric(df[columna_barri], errors='coerce')
    df = df.dropna(subset=[columna_barri])
    df[columna_barri] = df[columna_barri].astype(int)
    return df[(df[columna_barri] >= 1) & (df[columna_barri] <= 73)]


In [21]:
# cargamos dimensiones y diccionarios
df_dims = pd.read_csv('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\pad_dimensions.csv')


dicc_barris = df_dims[df_dims['Desc_Dimensio'] == 'CODI_BARRI_DEST'][['Codi_Valor', 'Desc_Valor_CA']].copy()
dicc_barris.rename(columns={'Codi_Valor': 'Codi_Barri', 'Desc_Valor_CA': 'Nom_Barri_Oficial'}, inplace=True)
dicc_barris['Codi_Barri'] = pd.to_numeric(dicc_barris['Codi_Barri'], errors='coerce')


dicc_educa = df_dims[df_dims['Desc_Dimensio'] == 'NIV_EDUCA_esta'][['Codi_Valor', 'Desc_Valor_CA']].copy()
dicc_educa.rename(columns={'Codi_Valor': 'NIV_EDUCA_esta', 'Desc_Valor_CA': 'Desc_Educa'}, inplace=True)
dicc_educa['NIV_EDUCA_esta'] = pd.to_numeric(dicc_educa['NIV_EDUCA_esta'], errors='coerce')


In [22]:

# 1. VUTs (Viviendas de Uso Turístico)

rutas_huts = sorted(glob.glob('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\huts\\*.csv'))
lista_df_huts = []

for ruta in rutas_huts:
    df_temp = pd.read_csv(ruta, encoding='latin1', on_bad_lines='skip', low_memory=False)
    df_temp['Any'] = int(os.path.basename(ruta)[0:4])
    
    # unificamos nombres de columnas
    df_temp.rename(columns={'BARRI': 'Nom_Barri', 'NOM_BARRI': 'Nom_Barri', 'CODI_BARRI': 'Codi_Barri'}, inplace=True)
    
    if 'NUMERO_PLACES' not in df_temp.columns:
        df_temp['NUMERO_PLACES'] = 0
    else:
        df_temp['NUMERO_PLACES'] = pd.to_numeric(df_temp['NUMERO_PLACES'], errors='coerce').fillna(0)

    cols = [c for c in ['Any', 'N_EXPEDIENT', 'NUMERO_PLACES', 'Nom_Barri', 'Codi_Barri'] if c in df_temp.columns]
    lista_df_huts.append(df_temp[cols])

df_huts_raw = pd.concat(lista_df_huts, ignore_index=True)

# nos quedamos con la última foto del año 
df_huts_raw = df_huts_raw.drop_duplicates(subset=['Any', 'N_EXPEDIENT'], keep='last')

# mapeamos los códigos de barrio faltantes 
if 'Codi_Barri' in df_huts_raw.columns and 'Nom_Barri' in df_huts_raw.columns:
    df_huts_raw['Nom_Barri'] = df_huts_raw['Nom_Barri'].astype(str).str.upper().str.strip()
    mapeo = df_huts_raw.dropna(subset=['Codi_Barri', 'Nom_Barri']).drop_duplicates(subset=['Nom_Barri'])
    dicc_temp = dict(zip(mapeo['Nom_Barri'], mapeo['Codi_Barri']))
    df_huts_raw['Codi_Barri'] = df_huts_raw['Codi_Barri'].fillna(df_huts_raw['Nom_Barri'].map(dicc_temp))

# estandarizamos y agrupamos
df_huts_raw = estandarizar_codi_barri(df_huts_raw, 'Codi_Barri')
df_huts_final = df_huts_raw.groupby(['Any', 'Codi_Barri']).agg(
    Total_HUTs=('N_EXPEDIENT', 'count'),
    Places_HUTs=('NUMERO_PLACES', 'sum')
).reset_index()

In [23]:

# 2. PRECIO DEL ALQUILER

df_lloguer_raw = pd.read_csv('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\Tabla estadística.csv')

# pasamos de formato ancho a largo, filtrando solo por barrios
df_lloguer_melted = df_lloguer_raw[df_lloguer_raw['Tipo de territorio'] == 'Barri'].melt(
    id_vars=['Territorio', 'Tipo de territorio'], var_name='Trimestre_str', value_name='Preu_Lloguer_m2'
)

# limpiamos y estandarizamos datos
df_lloguer_melted['Preu_Lloguer_m2'] = pd.to_numeric(df_lloguer_melted['Preu_Lloguer_m2'].astype(str).str.replace(',', '.').replace('-', np.nan), errors='coerce')
df_lloguer_melted['Any'] = df_lloguer_melted['Trimestre_str'].str[-4:].astype(int)

# calculamos media anual por barrio
df_lloguer_grouped = df_lloguer_melted.groupby(['Any', 'Territorio'])['Preu_Lloguer_m2'].mean().reset_index()

# mergeamos con el diccionario para sacar el Codi_Barri
df_lloguer_final = pd.merge(df_lloguer_grouped, dicc_barris, left_on='Territorio', right_on='Nom_Barri_Oficial', how='inner')
df_lloguer_final = estandarizar_codi_barri(df_lloguer_final, 'Codi_Barri')[['Any', 'Codi_Barri', 'Preu_Lloguer_m2']]



In [24]:
# 3. NUM CONTRATOS ANUALES

# cargamos excel
df_contractes_raw = pd.read_excel('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\anual_bcn_contractes.xlsx', skiprows=3)
df_contractes_raw.rename(columns={'Codi': 'Codi_Barri_CSV', 'Unnamed: 1': 'Territori'}, inplace=True)

# recortamos dataframe para aislar sección de barrios
idx_barris = df_contractes_raw[df_contractes_raw['Territori'].astype(str).str.strip().str.startswith('Barris')].index[0]
df_contractes_barris = df_contractes_raw.iloc[idx_barris+1:].dropna(subset=['Codi_Barri_CSV']).copy()

# pasamos de formato ancho a largo y limpiamos 
df_contractes_melted = df_contractes_barris.melt(id_vars=['Codi_Barri_CSV', 'Territori'], var_name='Any_str', value_name='Num_Contractes')
df_contractes_melted['Any'] = pd.to_numeric(df_contractes_melted['Any_str'], errors='coerce')
df_contractes_melted['Num_Contractes'] = pd.to_numeric(df_contractes_melted['Num_Contractes'], errors='coerce')
df_contractes_melted = df_contractes_melted.dropna(subset=['Num_Contractes', 'Any'])

# estandarizamos el código de barrio
df_contractes_melted = estandarizar_codi_barri(df_contractes_melted, 'Codi_Barri_CSV')

# mergeamos con el diccionario y filtramos
df_contractes_final = pd.merge(df_contractes_melted, dicc_barris, left_on='Codi_Barri_CSV', right_on='Codi_Barri', how='inner')
df_contractes_final = df_contractes_final[['Any', 'Codi_Barri', 'Num_Contractes']].astype(int)

In [25]:
# 4. CALCULO POBLACIÓN

# cargamos y concatenamos todos los csv de la carpeta
df_naix_raw = pd.concat((pd.read_csv(f) for f in glob.glob('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\lloc_naix\\*_pad_mdbas_lloc-naix_sexe.csv')), ignore_index=True)

# extraemos el año de la fecha y limpiamos la columna de valores
df_naix_raw['Any'] = df_naix_raw['Data_Referencia'].astype(str).str[0:4].astype(int)
df_naix_raw['Valor'] = pd.to_numeric(df_naix_raw['Valor'], errors='coerce').fillna(0)

# estandarizamos el código de barrio y agrupamos por lugar de nacimiento
df_naix = estandarizar_codi_barri(df_naix_raw, 'Codi_Barri')
df_naix_agg = df_naix.groupby(['Any', 'Codi_Barri', 'LLOC_NAIX'])['Valor'].sum().reset_index()

# calculamos la población total y filtramos la extranjera (códigos 4 y 5)
pob_total = df_naix_agg.groupby(['Any', 'Codi_Barri'])['Valor'].sum().reset_index(name='Poblacio_Total')
pob_ext = df_naix_agg[df_naix_agg['LLOC_NAIX'].isin([4, 5])].groupby(['Any', 'Codi_Barri'])['Valor'].sum().reset_index(name='Poblacio_Estrangera')

# unimos los totales, calculamos el porcentaje de extranjeros y filtramos columnas
df_origen_final = pd.merge(pob_total, pob_ext, on=['Any', 'Codi_Barri'], how='left').fillna({'Poblacio_Estrangera': 0})
df_origen_final['Pct_Estrangers'] = (df_origen_final['Poblacio_Estrangera'] / df_origen_final['Poblacio_Total']) * 100
df_origen_final = df_origen_final[['Any', 'Codi_Barri', 'Poblacio_Total', 'Pct_Estrangers']]

In [26]:
# 5. CÁLCULO UNIVERSITARIOS
# seguimos la misma lógica anterior

df_estudis_raw = pd.concat((pd.read_csv(f) for f in glob.glob('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\niv_educa\\*_pad_mdbas_niv-educa-esta_sexe.csv')), ignore_index=True)

df_estudis_raw['Any'] = df_estudis_raw['Data_Referencia'].astype(str).str[0:4].astype(int)
df_estudis_raw['Valor'] = pd.to_numeric(df_estudis_raw['Valor'], errors='coerce').fillna(0)

df_estudis_raw = estandarizar_codi_barri(df_estudis_raw, 'Codi_Barri')
df_estudis_agg = df_estudis_raw.groupby(['Any', 'Codi_Barri', 'NIV_EDUCA_esta'])['Valor'].sum().reset_index()

pob_total_est = df_estudis_agg.groupby(['Any', 'Codi_Barri'])['Valor'].sum().reset_index(name='Poblacio_Total')
pob_univ = df_estudis_agg[df_estudis_agg['NIV_EDUCA_esta'] == 5].groupby(['Any', 'Codi_Barri'])['Valor'].sum().reset_index(name='Poblacio_Universitaria')

df_estudis_final = pd.merge(pob_total_est, pob_univ, on=['Any', 'Codi_Barri'], how='left').fillna({'Poblacio_Universitaria': 0})
df_estudis_final['Pct_Universitaris'] = (df_estudis_final['Poblacio_Universitaria'] / df_estudis_final['Poblacio_Total']) * 100
df_estudis_final = df_estudis_final[['Any', 'Codi_Barri', 'Pct_Universitaris']]

In [27]:
# 6. NUM LOCALES ACTIVOS

llista_df_agg = []

# iteramos sobre los csv y extraemos el año del nombre del archivo
for archivo in glob.glob('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\cens_comercial\\*.csv'):
    match = re.search(r'(20\d{2})', archivo)
    if not match: continue
    
    # leemos el archivo
    df_temp = pd.read_csv(archivo, low_memory=False)
    
    # unificamos el filtro de locales activos según el esquema de cada año
    if 'CBARRI' in df_temp.columns:
        df_temp['Codi_Barri'] = df_temp['CBARRI']
        df_actius = df_temp[df_temp['BCN1'].astype(str).str.upper() != 'LOCAL BUIT'].copy()
    elif 'ID_PRINCIP' in df_temp.columns:
        df_actius = df_temp[df_temp['ID_PRINCIP'] == 1].copy()
    elif 'Codi_Principal_Activitat' in df_temp.columns:
        df_actius = df_temp[df_temp['Codi_Principal_Activitat'] == 1].copy()
    else: continue

    # estandarizamos el código de barrio y contamos el total de locales por barrio
    df_actius = estandarizar_codi_barri(df_actius, 'Codi_Barri')
    df_agg = df_actius.groupby('Codi_Barri').size().reset_index(name='Num_Locals_Actius')
    
    # asignamos el año correspondiente y lo añadimos a la lista
    df_agg['Any'] = int(match.group(1))
    llista_df_agg.append(df_agg)

# concatenamos todos los años y filtramos columnas finales
df_locals_final = pd.concat(llista_df_agg, ignore_index=True)[['Any', 'Codi_Barri', 'Num_Locals_Actius']]

In [28]:
# 7. RENTA BRUTA MEDIA

# cargamos y concatenamos todos los csv de la carpeta
df_renda_raw = pd.concat((pd.read_csv(f) for f in glob.glob('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\renda\\*.csv')), ignore_index=True)

# estandarizamos el código de barrio y forzamos a formato numérico
df_renda_raw = estandarizar_codi_barri(df_renda_raw, 'Codi_Barri')
df_renda_raw['Import_Renda_Bruta_€'] = pd.to_numeric(df_renda_raw['Import_Renda_Bruta_€'], errors='coerce')

# calculamos la media anual por barrio
df_renda_final = df_renda_raw.groupby(['Any', 'Codi_Barri'])['Import_Renda_Bruta_€'].mean().round(2).reset_index()

# renombramos la columna al formato final
df_renda_final.rename(columns={'Import_Renda_Bruta_€': 'Renda_Bruta_Mitjana'}, inplace=True)

In [29]:
# 8. MASTER DATASET 
# unimos todos los datasets

# Cross join para generar base 
df_master = pd.DataFrame({'Any': range(2014, 2025)}).merge(dicc_barris[['Codi_Barri', 'Nom_Barri_Oficial']].drop_duplicates(), how='cross')

# unimos todos los datasets ya limpios mediante left join  
df_master = pd.merge(df_master, df_huts_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_origen_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_estudis_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_lloguer_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_locals_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_renda_final, on=['Any', 'Codi_Barri'], how='left')
df_master = pd.merge(df_master, df_contractes_final, on=['Any', 'Codi_Barri'], how='left')

# imputamos nulos 
df_master.fillna({
    'Total_HUTs': 0, 'Places_HUTs': 0, 'Poblacio_Total': 0, 'Pct_Estrangers': 0,
    'Pct_Universitaris': 0, 'Preu_Lloguer_m2': 0, 'Num_Locals_Actius': 0, 'Renda_Bruta_Mitjana': 0,
    'Num_Contractes': 0
}, inplace=True)

# forzamos tipos de datos correctos
df_master = df_master.astype({
    'Total_HUTs': int,
    'Places_HUTs': int,
    'Poblacio_Total': int,
    'Preu_Lloguer_m2': float,
    'Num_Locals_Actius': int
})

# renombramos columnas
df_master.rename(columns={
    'Any': 'anyo', 'Codi_Barri': 'cod_barrio', 'Nom_Barri_Oficial': 'nom_barrio',
    'Total_HUTs': 'total_VUTs', 'Places_HUTs': 'plazas_VUTs', 'Poblacio_Total': 'poblacion_total',
    'Pct_Estrangers': 'pct_extranjeros', 'Pct_Universitaris': 'pct_universitarios',
    'Preu_Lloguer_m2': 'precio_alquiler_m2', 'Num_Locals_Actius': 'num_locales_activos',
    'Renda_Bruta_Mitjana': 'renta_bruta_media', 'Num_Contractes': 'num_contratos'
}, inplace=True)

df_master.sort_values(by=['anyo', 'cod_barrio'], inplace=True, ignore_index=True)

# exportamos
df_master.to_csv('master_dataset_bcn.csv', sep=';', decimal=',', encoding='utf-8-sig', index=False)
df_master.to_parquet('master_dataset_bcn.parquet', index=False)

df_master.head()

,anyo,cod_barrio,nom_barrio,total_VUTs,plazas_VUTs,poblacion_total,pct_extranjeros,pct_universitarios,precio_alquiler_m2,num_locales_activos,renta_bruta_media,num_contratos
0,2014,1,el Raval,0,0,0,0.0,0.0,10.475,2297,0.0,1583
1,2014,2,el Barri Gòtic,0,0,0,0.0,0.0,10.550,1936,0.0,630
2,2014,3,la Barceloneta,0,0,0,0.0,0.0,15.125,639,0.0,566
3,2014,4,"Sant Pere, Santa Caterina i la Ribera",0,0,0,0.0,0.0,11.225,1680,0.0,982
4,2014,5,el Fort Pienc,0,0,0,0.0,0.0,9.975,938,0.0,910


In [30]:
# 9. GEODATAFRAME PARA MAPAS

df_barris = pd.read_csv('C:\\Users\\alba_\\OneDrive\\Documentos\\1-MASTER\\1TFM\\datasets\\BarcelonaCiutat_Barris.csv')
df_barris['codi_barri'] = df_barris['codi_barri'].astype(int)

# unimos el master dataset con los datos espaciales
df_merged = pd.merge(df_master, df_barris, left_on='cod_barrio', right_on='codi_barri', how='left')

# convertimos geometry en objetos espaciales procesables
df_merged['geometry'] = df_merged['geometria_wgs84'].apply(wkt.loads)

# creamos el geodataframe asignando la columna geometry y el sistema de coordenadas
gdf_master = gpd.GeoDataFrame(df_merged, geometry='geometry', crs="EPSG:4326")

# exportamos a geojson
gdf_master.to_file("gdf_master.geojson", driver="GeoJSON")